# Sentence-End Detection — Feature Analysis & Model Training

Фичи извлечены **Swift/vDSP кодом** (идентично инференсу).
Здесь: XGBoost + SHAP на GPU → какие фичи реально важны → идут в Swift студент-модель.

**Очистка меток**: y=1 оставляем только если есть акустическое подтверждение:
- `pause_sec >= 50ms` (детектируемая пауза)
- `energy_slope < -0.001` (энергия падает)
- `rms_ratio > 1.2` (речь после тише)

Без очистки 58.8% конца предложений имеют паузу < 50ms — шум транскриптора.

In [ ]:
!pip install -q xgboost shap scikit-learn pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings, subprocess
warnings.filterwarnings('ignore')

gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'CPU only')
DEVICE = 'cuda' if gpu else 'cpu'

## 1. Загрузка данных (акустически очищенные метки)

In [ ]:
import os, urllib.request

REPO = 'https://raw.githubusercontent.com/russianoracle/sentence-end-training/main'

# Clean CSV (acoustic label filtering: pause|slope|rms confirmed only)
# Raw: sentence_end_swift_features.csv (pos=13.2%, XGB AUC=0.629)
# Clean: sentence_end_swift_features_clean.csv (pos=10.5%, XGB AUC=0.718)
CSV = 'sentence_end_swift_features_clean.csv'

if not os.path.exists(CSV):
    print(f'Downloading {CSV}...')
    urllib.request.urlretrieve(f'{REPO}/{CSV}', CSV)

df = pd.read_csv(CSV)
feature_cols = [c for c in df.columns if c not in ('y', 'source')]

print(f'Rows: {len(df):,}  |  features: {len(feature_cols)}  |  pos: {df["y"].mean()*100:.1f}%')
print(f'Features: {feature_cols}')
df.groupby('source')['y'].agg(['count','sum','mean']).rename(
    columns={'count':'total','sum':'pos_n','mean':'pos_rate'})

## 2. Per-feature AUC (что реально работает)

In [ ]:
X = df[feature_cols].fillna(0).values
y = df['y'].values

rows = []
for i, c in enumerate(feature_cols):
    auc = roc_auc_score(y, X[:, i])
    if auc < 0.5: auc = 1 - auc
    rows.append({'feature': c, 'auc': auc,
                 'neg_mean': df[df['y']==0][c].mean(),
                 'pos_mean': df[df['y']==1][c].mean()})

feat_df = pd.DataFrame(rows).sort_values('auc', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_df['feature'], feat_df['auc'] - 0.5, left=0.5)
ax.axvline(0.5, color='black', lw=0.8)
ax.set_xlabel('AUC (single feature)')
ax.set_title('Swift feature separability (clean labels)')
for bar, val in zip(ax.patches, feat_df['auc']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()
feat_df

## 3. XGBoost (GPU) + SHAP

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pos_w = (y_tr==0).sum() / (y_tr==1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=pos_w,
    tree_method='hist', device=DEVICE,
    eval_metric='auc', random_state=42, verbosity=0,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
xgb_prob = xgb_model.predict_proba(X_te)[:,1]
auc = roc_auc_score(y_te, xgb_prob)
print(f'XGBoost AUC: {auc:.4f}  (device={DEVICE})')

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_te)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_te, feature_names=feature_cols, show=False)
plt.title('SHAP — feature impact on sentence-end prediction (clean labels)')
plt.tight_layout(); plt.show()

mean_shap = np.abs(shap_values).mean(axis=0)
shap_df   = pd.DataFrame({'feature': feature_cols, 'mean_shap': mean_shap}) \
              .sort_values('mean_shap', ascending=False)
print(shap_df.to_string(index=False))

## 4. Model comparison: LR vs XGBoost, all features vs top-8 SHAP

In [ ]:
TOP_N = 8
top_features = shap_df['feature'].iloc[:TOP_N].tolist()
idx_top      = [feature_cols.index(f) for f in top_features]

print(f'Top-{TOP_N} фичей по SHAP (все акустические → идут в Swift):')
for i, f in enumerate(top_features, 1):
    sv = shap_df[shap_df['feature']==f]['mean_shap'].values[0]
    print(f'  {i}. {f:<25} {sv:.4f}')

results   = {}
probs_map = {}

# XGBoost все фичи
results['XGBoost all'] = roc_auc_score(y_te, xgb_prob)
probs_map['XGBoost all'] = xgb_prob

# XGBoost топ-N
m2 = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
     scale_pos_weight=pos_w, tree_method='hist', device=DEVICE, random_state=42, verbosity=0)
m2.fit(X_tr[:,idx_top], y_tr)
p = m2.predict_proba(X_te[:,idx_top])[:,1]
results[f'XGBoost top-{TOP_N}'] = roc_auc_score(y_te, p)
probs_map[f'XGBoost top-{TOP_N}'] = p

# LR все фичи
sc = StandardScaler()
lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
lr.fit(sc.fit_transform(X_tr), y_tr)
lr_prob = lr.predict_proba(sc.transform(X_te))[:,1]
results['LR all'] = roc_auc_score(y_te, lr_prob)
probs_map['LR all'] = lr_prob

# LR топ-N
sc2 = StandardScaler()
lr2 = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
lr2.fit(sc2.fit_transform(X_tr[:,idx_top]), y_tr)
p = lr2.predict_proba(sc2.transform(X_te[:,idx_top]))[:,1]
results[f'LR top-{TOP_N}'] = roc_auc_score(y_te, p)
probs_map[f'LR top-{TOP_N}'] = p

print('\n=== AUC ===')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k:<25} {v:.4f}  ' + '█' * int((v-0.5)*200))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
names = list(results.keys()); aucs = [results[n] for n in names]
ax1.barh(names, aucs); ax1.axvline(0.5, color='red', ls='--', alpha=0.4)
ax1.set_xlim(0.45, 0.80); ax1.set_xlabel('AUC-ROC'); ax1.set_title('Model comparison')
for i, v in enumerate(aucs): ax1.text(v+0.003, i, f'{v:.4f}', va='center')

for name, prob in probs_map.items():
    fpr, tpr, _ = roc_curve(y_te, prob)
    ax2.plot(fpr, tpr, label=f'{name} ({results[name]:.3f})')
ax2.plot([0,1],[0,1],'k--',alpha=0.3)
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR'); ax2.set_title('ROC Curves'); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Экспорт весов LR (top-8 фичей) для Swift

In [ ]:
import json

swift_model = {
    'coef':      lr2.coef_[0].tolist(),
    'intercept': float(lr2.intercept_[0]),
    'scaler': {
        'mean':  sc2.mean_.tolist(),
        'scale': sc2.scale_.tolist(),
    },
    'features': top_features,
    'auc_roc':  float(results[f'LR top-{TOP_N}']),
    'dataset':  CSV,
}

with open('sentence_end_model_colab.json', 'w') as f:
    json.dump(swift_model, f, indent=2)

print('Saved: sentence_end_model_colab.json')
print(f'AUC:      {swift_model["auc_roc"]:.4f}')
print(f'Features: {top_features}')
print(f'Coef:     {[round(c,4) for c in swift_model["coef"]]}')

try:
    from google.colab import files
    files.download('sentence_end_model_colab.json')
except:
    pass